# Идеи: анализ и подготовка данных

Мы с вами уже многое попробовали на курсе — от знакомства с таблицами до обучения моделей. Здесь собраны небольшие подсказки, которые помогут вспомнить знакомые приёмы и придумать свои эксперименты.

Это **необязательная рекомендация и небольшая наводка на мысль**. Не нужно выполнять всё именно так: вы можете обрабатывать данные совершенно по-разному. Если захотите попробовать новые методы, которых не было на курсе, это тоже очень приветствуется! Главное — разобраться, как они работают, и проверить свою идею.

Ниже есть короткие примеры и места `TODO` для вашего кода. Можно выбрать только интересные вам шаги. Подробное описание полей лежит в `README.md` в папке `final_project/data`. Примеры анализа данных можно найти в `lesson_2` репозитория курса, регрессии — в `lesson_3`, классификации — в `lesson_4`, приложения — в отдельном открытом репозитории `streamlit-app`, где размещён код уже развёрнутого приложения.

Начнём с исследования train. В ноутбуках обучения будем учить модели на train и сравнивать их на test.

### Как найти места для вашей работы

В ноутбуке есть пометки `TODO` — в коде они выглядят как `# TODO`. Попробуйте поиск по слову `TODO`: так будет проще пройти все шаги и ничего не упустить. Перед завершением ещё раз просмотрите найденные места.

Код дописывайте в кодовых ячейках, а наблюдения и объяснения — в Markdown под строкой `Ответ:`.

В ноутбуке используются `assert` — это готовые проверки, которые помогают убедиться, что данные и результаты имеют ожидаемый вид. В самих проверках ничего дописывать не нужно: просто запускайте их вместе с остальными ячейками. Если проверка сообщает об ошибке, вернитесь к соответствующему шагу и проверьте свой код.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Откройте ноутбук из final_project или final_project/notebooks.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Откройте ноутбук из папки final_project."

regression = pd.read_csv(ROOT / "data" / "regression.csv")
classification = pd.read_csv(ROOT / "data" / "classification.csv")

# Для первых наблюдений возьмём обучающие строки.
train = regression.loc[regression["split"] == "train"].copy()
class_train = classification.loc[classification["split"] == "train"].copy()
train.head()

## 1. Посмотрим на таблицу целиком

Какие столбцы числовые, а какие категориальные? Где больше пропусков? Есть ли одинаковые ID? Начните с нескольких знакомых команд, затем добавьте то, что хочется проверить вам.

In [ ]:
train.info()

In [ ]:
missing_share = train.isna().mean().sort_values(ascending=False)
missing_share

In [ ]:
# TODO: посмотрите describe() для чисел и частоты интересных категорий.
# TODO: проверьте повторы listing_id через duplicated().sum().

**TODO:** Запишите 2–3 наблюдения: что стоит учесть при подготовке модели?

Ответ:

## 2. `city`: знакомимся с городами

Посмотрите, одинаково ли представлены города. Лишние пробелы можно убрать через `.str.strip()`. Ниже мы только посмотрим на результат — исходная таблица останется прежней.

Для модели можно попробовать One-Hot Encoding. Необязательно назначать городам числа: такой порядок может оказаться бессмысленным.

In [ ]:
train["city"].value_counts(dropna=False)

In [ ]:
city_example = train["city"].astype("string").str.strip()
# TODO: сравните исходные и очищенные значения.
# При необходимости попробуйте .str.lower() и удаление повторяющихся пробелов.

## 3. `district`: что делать с районом

У района есть пропуски. Можно попробовать отдельную категорию «неизвестно», а затем сравнить модель с районом и без него. Это пример идеи, а не обязательный способ заполнения.

Одинаковое название района в двух городах может обозначать разные места. Как вы могли бы это учесть?

In [ ]:
print("Доля пропусков района:", train["district"].isna().mean())
district_example = train["district"].fillna("неизвестно")
district_example.value_counts().head(10)

In [ ]:
# TODO: попробуйте объединить город и район в новый признак.
# Подсказка: строки можно соединять через +, например city + " | " + district.
# Пока работайте с копией: district_data = train.copy().

## 4. `street`: улицы, редкие значения и новые категории

У улиц могут отличаться написания, а некоторые встречаются очень редко. Посмотрите на частоты и подумайте, стоит ли объединять редкие значения. Не спешите заменять `улица` на `ул.` по всей таблице: сначала проверьте примеры, чтобы не объединить разные места.

Для редких категорий частоты считаем только на train. Ниже — начало такого эксперимента; порог вы можете выбрать сами. В модели пригодится encoder, который принимает и незнакомую улицу.

In [ ]:
street_example = train["street"].fillna("неизвестно").str.strip()
# Учтём город: одинаковое название улицы может встречаться в разных городах.
city_street = train["city"].fillna("неизвестно").str.strip() + " | " + street_example
street_counts = city_street.value_counts()
street_counts.head(10)

In [ ]:
minimum_count = 5  # Это только пример — можно исследовать другие значения.
rare_streets = street_counts[street_counts < minimum_count].index
# TODO: попробуйте .where() или .isin(), чтобы выделить редкие улицы.
# Выбранное по train правило нужно одинаково применять к остальным частям.

**TODO:** Как вы предлагаете обрабатывать улицу, которой не было в train? Кратко объясните свою идею.

Ответ:

## 5. `area`: площадь и её распределение

Есть ли квартиры, которые заметно отличаются по площади? Как связаны площадь и цена? Выброс на графике — повод разобраться, а не обязательно удалить строку.

Для линейной модели можно сравнить вариант с масштабированием и без него. Если используете масштабирование, вычислите его параметры только на train и примените те же параметры к test.

In [ ]:
train["area"].hist(bins=30)
plt.xlabel("Площадь, м²")
plt.ylabel("Число объявлений")
plt.show()
# TODO: постройте scatter площади и price_rub на train.
# Подсказка: train.plot.scatter(x="area", y="price_rub").

## 6. `rooms`: комнаты и студии

Вспомните, что 0 здесь означает студию, а пропуск — неизвестное количество комнат. Это разные ситуации. Попробуйте посмотреть частоты и сравнить числовую и категориальную обработку.

Если выбираете заполнение медианой, её нужно вычислить только по train. Ниже — маленький пример. Сохраните найденную медиану: она пригодится для заполнения пропусков на test и в приложении.

In [ ]:
train["rooms"].value_counts(dropna=False).sort_index()

In [ ]:
rooms_median = train["rooms"].median()
rooms_example = train["rooms"].fillna(rooms_median)
# TODO: посмотрите, сколько пропусков было заполнено.

**TODO:** Нужна ли модели отдельная информация о том, что количество комнат неизвестно? Как вы думаете, чем это может помочь?

Ответ:

## 7. `floor` и `floors_total`: исследуем этажи

Можно проверить согласованность этажей и придумать новые признаки: первый этаж, последний этаж или относительный этаж. Выберите то, что хочется проверить вам.

В примере ниже новый признак пока создаётся только на копии train. Если решите его использовать, добавьте одинаковое правило и для новых квартир в приложении.

In [ ]:
impossible_floor = train["floor"] > train["floors_total"]
train.loc[impossible_floor, ["floor", "floors_total"]].head()

In [ ]:
floor_example = train[["floor", "floors_total"]].copy()
# where оставит пропуск вместо деления на ноль, если такое значение встретится.
denominator = floor_example["floors_total"].where(floor_example["floors_total"] > 0)
floor_example["floor_ratio"] = floor_example["floor"] / denominator
# TODO: добавьте is_first_floor и is_top_floor.
# Подсказка: сравнение через == возвращает True/False; .astype(int) даст 1/0.
floor_example.head()

## 8. `house_type`: категории типов домов

Посмотрите на типы домов и пропуски. Здесь можно попробовать заполнение отдельной категорией и One-Hot Encoding. Если захотите объединить похожие названия, сначала проверьте, действительно ли у них один смысл.

In [ ]:
train["house_type"].value_counts(dropna=False)
# TODO: предложите способ заполнения пропусков и обработки категорий.
# Попробуйте .str.strip() на копии столбца и сравните набор значений.

## 9. `market_type`: первичный и вторичный рынок

Здесь тоже начнём с реальных значений. После этого можно выбрать OHE или явный словарь кодирования. Если используете словарь, подумайте, как поступить с пропуском или новым значением.

In [ ]:
train["market_type"].value_counts(dropna=False)
# TODO: если хотите, составьте свой словарь и попробуйте .map().
# Перед этим проверьте написания категорий в таблице.

## 10. `lift_count`: лифты и неизвестные значения

Отсутствие записи о лифте не означает, что лифта нет. Посмотрите, как часто встречаются пропуски. Можно попробовать заполнение, индикатор пропуска или их сочетание — и сравнить результат на test.

In [ ]:
train["lift_count"].value_counts(dropna=False).sort_index()

In [ ]:
lift_example = train[["lift_count"]].copy()
lift_example["lift_missing"] = lift_example["lift_count"].isna().astype(int)
# TODO: выберите способ заполнения lift_count.
# Если он использует статистику, вычислите её только на train.
lift_example.head()

## 11. `listing_id`, `group_id`, `split`: оставим полезную служебную информацию

Эти поля помогут сопоставлять квартиры и сохранять разбиение. Модели передавать их не нужно, но в новых CSV их лучше оставить. Это позволит использовать привычные ноутбуки обучения и проверить, что связанные объявления не разошлись по разным частям.

In [ ]:
service_columns = ["listing_id", "group_id", "split"]
assert regression.groupby("group_id")["split"].nunique().max() == 1
regression["split"].value_counts()
# TODO: перед обучением соберите свой список features без служебных полей.

## 12. `price_rub`: цель регрессии и признак классификации

Для дорогих квартир ошибки могут быть особенно велики. Посмотрите на распределение цены. По желанию можно попробовать логарифм цели в ноутбуке регрессии; тогда прогноз обязательно нужно преобразовать обратно в рубли.

В этом ноутбуке оставим исходный `price_rub` в рублях, чтобы смысл сохраняемых данных не менялся. В классификации общая цена — входной числовой признак; её тоже можно масштабировать, вычислив параметры только на train. В регрессии цена остаётся целью и во входы не включается.

In [ ]:
train["price_rub"].hist(bins=30)
plt.xlabel("Цена, руб.")
plt.ylabel("Число объявлений")
plt.show()
# TODO: посмотрите медиану и квантили цены на train через median() и quantile().

## 13. `above_reference`: знакомимся с балансом классов

Метка `above_reference` уже подготовлена, её смысл мы обсуждали на занятии. Посмотрите, одинаково ли часто встречаются классы. Для прогноза класса в ноутбуке классификации используем фиксированный порог вероятности 0.5.

Здесь цель оставим без изменений. Общая цена `price_rub` уже есть в классификационном CSV и входит в признаки. Посмотрите, как подготовить её вместе с остальными числовыми полями.

In [ ]:
class_train["above_reference"].value_counts(normalize=True)
# TODO: попробуйте показать баланс классов на столбчатом графике.

## 14. Соберём выбранные идеи вместе

Необязательно использовать все примеры выше. Выберите несколько идей и перенесите их в функцию ниже. Пока она просто возвращает копию таблицы — это место для вашего решения.

В неё удобно включить правила, которые не нужно обучать: очистку текста, сочетание города и района или этажные признаки. Медианы, частоты редких категорий и параметры кодирования и масштабирования определяйте только по train. Сохраните их в словарь и применяйте те же значения к test и новым квартирам.

Для приложения правило обработки должно быть таким же. Запишите выбранные шаги и повторите их в ноутбуках обучения и в `app.py` с теми же сохранёнными значениями.

In [ ]:
def prepare_data(frame):
    result = frame.copy()
    # TODO: добавьте выбранные вами правила.
    # Например: очистку текста или новый этажный признак.
    # Сохраняйте строки, служебные поля и исходную целевую переменную.
    return result


regression_prepared = prepare_data(regression)
classification_prepared = prepare_data(classification)

## 15. Проверим, что сохранили важное

Эти короткие проверки помогут убедиться, что мы не потеряли квартиры, не изменили цели и сохранили разбиение. Для исследования новых признаков здесь не задан один обязательный результат.

Если вы осознанно решите отбирать строки, отдельно объясните правило и согласуйте его для обеих задач. Тогда проверку числа строк нужно будет адаптировать к вашему решению.

In [ ]:
for original, prepared, target in [
    (regression, regression_prepared, "price_rub"),
    (classification, classification_prepared, "above_reference"),
]:
    assert len(prepared) == len(original), "Проверьте, какие строки были удалены."
    assert prepared[service_columns].equals(original[service_columns])
    assert prepared[target].equals(original[target]), "Проверьте, не изменилась ли цель."
    assert prepared.groupby("group_id")["split"].nunique().max() == 1

## 16. Сохраним ваши версии датасетов

Когда выберете нужные изменения, можно записать новые таблицы в CSV. Для этого у pandas есть функция **`DataFrame.to_csv()`**. Параметр `index=False` не добавляет в файл лишний столбец индекса.

Советую сначала сохранить свои варианты рядом с исходными: например, `regression_prepared.csv` и `classification_prepared.csv` в папке `final_project/data`. Если захотите именно перезаписать исходные датасеты, укажите имена `regression.csv` и `classification.csv`, предварительно сохранив их копии.

Ниже команды оставлены в комментариях: раскомментируйте их, когда будете готовы. Затем откройте ноутбуки обучения и укажите новые имена файлов в `pd.read_csv()`. Если добавили признаки, включите нужные поля в `features` и настройте их обработку.

А теперь самое интересное: сравните test-метрики разных вариантов и расскажите, помогли ли ваши идеи. Помните, что после выбора лучшего варианта по test это уже учебное сравнение, а не независимая итоговая оценка.

In [ ]:
# Раскомментируйте, когда будете готовы сохранить свои изменения:
# regression_prepared.to_csv(ROOT / "data" / "regression_prepared.csv", index=False, encoding="utf-8-sig")
# classification_prepared.to_csv(ROOT / "data" / "classification_prepared.csv", index=False, encoding="utf-8-sig")